# Calibration gradient-free — CMA-ES vs Optax

This page runs the same multi-start driver with two backends: Optax (Adam) and
CMA-ES (evosax). CMA-ES does not need gradients; each step is one generation
and costs `population × starts` model solves.

The claim to check:

1. Both backends' **mean best loss falls** over chunks on the same starts.
2. CMA-ES fitted infection rates sit within 0.05 of the true rate.


In [ ]:
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
    derived_refs,
)
from summer4.epi.calibration import (
    BayesianModel,
    NormalLikelihood,
    Uniform,
    workflow as wf,
)


## Shared SIR design

Score an LHS and keep four starts for both backends.


In [ ]:
class Rates(NamedTuple):
    infection: float
    recovery: float


TRUE_INFECTION = 0.35
times = np.array([0.0, 20.0, 40.0, 60.0])
state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
qty = Compartments(where=state["I"])
truth = cm.run(
    {"infection": TRUE_INFECTION, "recovery": 0.1},
    y0,
    t0=0.0,
    t1=80.0,
    dt=1.0,
    save=SavePlan(requests={"I": SaveRequest(qty, ts=times)}),
    solver="euler",
)
raw = truth["I"].at_times(times).values
obs = np.asarray(raw.data if hasattr(raw, "data") else raw).reshape(-1)
targets = TargetSet(
    targets=(
        Target(
            key="I",
            times=times,
            values=obs,
            quantity=qty,
            likelihood=NormalLikelihood(sd=5.0),
        ),
    )
)
bm = BayesianModel(
    cm,
    {"recovery": 0.1},
    priors=(Uniform("infection", 0.05, 1.0),),
    targets=targets,
    y0=y0,
    run_kwargs={"t0": 0.0, "t1": 80.0, "dt": 1.0, "solver": "euler"},
)
design = wf.evaluate(bm, wf.lhs(bm, 64, seed=0), batch_size=32)
starts = design.best(4)


## Optax and CMA-ES on the same starts

Optax uses gradient steps; CMA-ES uses population=8 generations. Plot mean
loss per chunk for each backend (against generation/chunk index).


In [ ]:
optax_res = wf.optimize(
    bm,
    starts,
    method=wf.Optax(learning_rate=0.05),
    tuning=wf.AutoTune(probe_steps=10, probe_starts=2, patience=2, rtol=1e-5),
    chunk_steps=25,
    max_steps=100,
    seed=0,
)
cma_res = wf.optimize(
    bm,
    starts,
    method=wf.CMAES(sigma0=0.3, population=8),
    tuning=wf.AutoTune(patience=3, rtol=1e-4, max_restarts=0),
    chunk_steps=5,
    max_steps=40,
    seed=0,
)

rows = []
for name, res in (("optax", optax_res), ("cmaes", cma_res)):
    trace = np.asarray(res.loss_trace)
    for i, mean_loss in enumerate(trace.mean(axis=1)):
        rows.append({"backend": name, "chunk": i, "mean_loss": float(mean_loss)})
df = pd.DataFrame(rows)
fig = df.plot(
    x="chunk",
    y="mean_loss",
    color="backend",
    title="Mean best loss per chunk: Optax vs CMA-ES",
)
fig.update_layout(xaxis_title="chunk", yaxis_title="mean best loss")
fig.show()

assert float(optax_res.loss_trace[-1].mean()) < float(optax_res.loss_trace[0].mean())
assert float(cma_res.loss_trace[-1].mean()) < float(cma_res.loss_trace[0].mean())
cma_fit = np.asarray(cma_res.candidates.params["infection"])
assert np.all(np.abs(cma_fit - TRUE_INFECTION) < 0.05), cma_fit
print("CMA-ES fitted", cma_fit, "Optax fitted", np.asarray(optax_res.candidates.params["infection"]))
